<a href="https://colab.research.google.com/github/Varpyx/Uro-Bootstrap/blob/main/fml_01_student_en_active_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fundamentals of Machine Learning — Project Part 1
## Základní zkoumání datasetu Spotify Songů
### Dataset dostupný zde: https://www.kaggle.com/datasets/yashdev01/spotify-tracks-dataset


Nejdříve si vypíšu tvar, a zjistim info o datovych typech. Dale se podivame kolik hodnot missuje, a vypišeme si prvních 5 řádků.

In [9]:
import os
import pandas as pd
import kagglehub
import glob

# Stáhnutí nejnovější verze datasetu z Kaggle
path = kagglehub.dataset_download("yashdev01/spotify-tracks-dataset")

# Automatické nalezení libovolného CSV souboru ve stažené složce
csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
if not csv_files:
    raise FileNotFoundError(f"V adresáři {path} nebyl nalezen žádný .csv soubor.")

# Načtení staženého CSV souboru
df = pd.read_csv(csv_files[0])

print(f"\nPočet řádků (skladeb): {df.shape[0]}")
print(f"Počet sloupců (vlastností): {df.shape[1]}\n")

df.info()

print("Chybějici hodnoty:\n")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Žádné chybějící hodnoty nebyly nalezeny.")

print("PRVNÍCH 5 ŘÁDKŮ")
df.head()

Using Colab cache for faster access to the 'spotify-tracks-dataset' dataset.

Počet řádků (skladeb): 114000
Počet sloupců (vlastností): 22

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0.1      114000 non-null  int64  
 1   Unnamed: 0        114000 non-null  int64  
 2   track_id          114000 non-null  object 
 3   artists           113999 non-null  object 
 4   album_name        113999 non-null  object 
 5   track_name        113999 non-null  object 
 6   popularity        114000 non-null  int64  
 7   duration_ms       114000 non-null  int64  
 8   explicit          114000 non-null  bool   
 9   danceability      114000 non-null  float64
 10  energy            114000 non-null  float64
 11  key               114000 non-null  int64  
 12  loudness          114000 non-null  float64
 13  mode              114000

,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


# Souhrn
Z vypisu dat vidime že unnamed colums jsou zde nejspíš stareé ID a nebudeme je tedy potřebovat.

Chybí nám 1 hodnota u artists, album_name a track_name. Tyto hodnoty nepoužijeme a smažeme

# Rozdělení sloupců:

## Metadata:
track_id, artists, album_name, track_name, track_genre

## Číselné příznaky (Audio Features):
danceability, energy, key, loudness, mode, speechiness, acousticness, instrumentalness, liveness, valence, tempo, duration_ms, popularity.

In [10]:
# 1. Odstranění zbytečných indexových sloupců
df_clean = df.drop(columns=["Unnamed: 0.1", "Unnamed: 0"])

# 2. Odstranění jednoho chybějícího řádku
df_clean = df_clean.dropna(subset=["artists", "album_name", "track_name"]).copy()

metadata_cols = ["track_id", "artists", "album_name", "track_name", "track_genre"]

feature_cols = [
    "danceability", "energy", "key", "loudness", "mode",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo", "duration_ms", "popularity"
]

# 4. Vytvoření samostatné matice pouze pro číselné příznaky
X = df_clean[feature_cols]

print(f"Počet řádků po vyčištění: {len(df_clean)}")
print(f"Počet vybraných číselných příznaků pro modelování: {X.shape[1]}")

Počet řádků po vyčištění: 113999
Počet vybraných číselných příznaků pro modelování: 13


## 0. Working with a notebook

A notebook contains two main types of cells:

- **Code cells** execute Python code.
- **Markdown cells** contain formatted text, instructions, equations, and interpretations.

Useful shortcuts:
| Action | Shortcut |
|---|---|
| Run a cell and move to the next one | `Shift + Enter` |
| Run a cell and stay in it | `Ctrl + Enter` |
| Insert a cell above | `A` in command mode |
| Insert a cell below | `B` in command mode |
| Delete a cell | `D`, `D` in command mode |

### Important: notebooks are stateful

Variables remain in memory after a cell has been executed. Therefore, the visible order of cells and the actual execution order may differ.

When results look suspicious, use:

- **Jupyter:** `Kernel → Restart Kernel and Run All Cells`
- **Google Colab:** `Runtime → Restart session and run all`

For this exercise, Google Colab is sufficient. A local Jupyter installation is optional.

### Predict before running

Read both code cells below without executing them.

1. What will the second cell print?
2. What happens if you run the second cell before the first one?
3. What changes if you execute the first cell again with a different value of `exercise_number`?

In [ ]:
course_name = "Fundamentals of Machine Learning"
exercise_number = 1

In [ ]:
print(f"{course_name}: Exercise {exercise_number}")

**Compare the observed result with your prediction.**  

## 1. Imports and display settings

We will use four data-analysis and visualisation libraries:

- **Pandas** — tabular data processing,
- **NumPy** — numerical operations,
- **Matplotlib** — low-level plotting and detailed control,
- **Seaborn** — convenient statistical visualisation built on Matplotlib,
- **Plotly Express** — interactive visualisation.

Plotly is pre-installed in Google Colab. In a local environment, install it with:

```python
%pip install plotly
```

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

## 2. Dataset

**Source:** [House Prices: Advanced Regression Techniques](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)

Each row represents one house sale. The original dataset contains many variables. We will use a smaller subset later.

Important variables include:

| Variable | Meaning |
|---|---|
| `Id` | Row/house identifier |
| `SalePrice` | Sale price in USD |
| `MSSubClass` | Building class code |
| `Neighborhood` | Physical location within Ames |
| `BldgType` | Type of dwelling |
| `HouseStyle` | Style of dwelling |
| `OverallQual` | Overall material and finish quality |
| `OverallCond` | Overall condition rating |
| `YearBuilt` | Original construction year |
| `YrSold` | Year of sale |
| `Heating` | Type of heating |
| `CentralAir` | Central air conditioning (`Y`/`N`) |
| `GrLivArea` | Above-ground living area in square feet |
| `LotArea` | Lot size in square feet |
| `BedroomAbvGr` | Bedrooms above basement level |

> `SalePrice` will later become a **target variable** in a regression task. Here, we only explore and manipulate the data.

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/rasvob/"
    "VSB-FEI-Fundamentals-of-Machine-Learning-Exercises/"
    "master/datasets/zsu_cv1_data.csv"
)

df_full = pd.read_csv(DATA_URL)

print("Dataset loaded successfully.")
print(f"Shape: {df_full.shape[0]} rows × {df_full.shape[1]} columns")

## 3. First look at the data

A reliable analysis begins by checking what one row represents, which variables are available, and whether the dataset looks as expected.

In [ ]:
df_full.head()

### Activity 1 — Choose the right inspection command

Match each question with a suitable command from the code bank.

**Questions**

A. What are the last ten observations?  
B. What do five reproducibly selected random observations look like?  
C. How many rows and columns are present?  
D. What are the column names?  
E. What are the data types and non-missing counts?

**Code bank**

```python
df_full.info()
df_full.shape
df_full.columns
df_full.tail(10)
df_full.sample(5, random_state=42)
```

Run the commands in an order that you consider useful. Add or rearrange cells as needed.

In [ ]:
# Start with the command that you consider most informative.
df_full.sample(5, random_state=42)

### Stop and explain

Discuss or write short answers:

1. What does one row represent?
2. Which column is the likely target variable for a future regression model?
3. Why should `Id` normally not be used as a numerical predictor?
4. Which inspection command would you run first on an unfamiliar dataset, and why?

## 4. Data types and missing values

In [ ]:
df_full.info()

### Activity 2 — Numeric does not always mean quantitative

The following variables are all stored as numbers:

- `Id`
- `MSSubClass`
- `OverallQual`
- `GrLivArea`

Assign each variable one of these roles:

- **identifier**
- **categorical code**
- **ordinal variable**
- **continuous measurement**

Then answer: what could go wrong if `MSSubClass` were interpreted as an ordinary continuous measurement?

In [ ]:
df_full[["Id", "MSSubClass", "OverallQual", "GrLivArea"]]

### Missing values: compare two approaches

First, predict which output will be more useful for an analyst.

In [ ]:
# Approach A
df_full.isna()

In [ ]:
# Approach B
missing_values = df_full.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0].head(15)

### Explain the difference

Why is Approach B usually more useful? What information is still missing if we only report the number of missing values and not their percentage?

### Modify the analysis

Extend Approach B so that the output contains both:

- the number of missing values,
- the percentage of missing values.

Do not ask an AI assistant for a complete solution immediately. First decide which denominator is required.

Hint: what is the output of `len(df_full)`

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_full.isna().sum(),
    # Add a second column named "missing_percent".

})

missing_summary = missing_summary.loc[
    missing_summary["missing_count"] > 0
].sort_values("missing_count", ascending=False)

missing_summary.head(15)

## 5. Create a smaller working DataFrame

In [ ]:
selected_columns = [
    "Id",
    "SalePrice",
    "MSSubClass",
    "Neighborhood",
    "BldgType",
    "HouseStyle",
    "OverallQual",
    "OverallCond",
    "YearBuilt",
    "YrSold",
    "Heating",
    "CentralAir",
    "GrLivArea",
    "LotArea",
    "BedroomAbvGr",
]

df = df_full.loc[:, selected_columns].copy()
df.head()

### What we say by using : in `df_full[:, selected_columns]`

### Why `.copy()`?

Predict what the intention of `.copy()` is. You do not need to memorise all technical details yet; explain why an independent working table may be safer than repeatedly modifying the original data.

## 6. Selecting rows and columns: `iloc` and `loc`

- `iloc` selects by **integer position**.
- `loc` selects by **labels** and can also use logical conditions.

### Activity 3 — Predict the shape

Without running the cells, predict the number of rows and columns in each result.

In [ ]:
selection_a = df.iloc[3:10, 2:6]
selection_a

In [ ]:
selection_b = df.loc[3:10, "MSSubClass":"OverallCond"]
selection_b

Now compare the actual shapes:

```python
selection_a.shape
selection_b.shape
```

Explain why the number of selected rows differs. Also explain why selecting columns by name may be safer than relying on their current positions.

### Debugging activity

The intention is to select the `LotArea` column for rows with labels 3 to 9. The code below does not express that intention correctly.

1. Identify the problem.
2. Correct it using `loc`.
3. Write an alternative using `iloc`.

In [ ]:
# Incorrect for the stated intention:
wrong_selection = df.iloc[3:9, "LotArea"]

## 7. Filtering with logical conditions

In [ ]:
# Worked example: houses built in 2000 or later
houses_2000_or_later = df.loc[df["YearBuilt"] >= 2000]
houses_2000_or_later.head()

### Activity 4 — Debug three filters

Each filter below contains a common error. Correct the expressions and explain the error in words.

The intended subsets are:

1. houses cheaper than 180,000 USD **and** with at least two bedrooms,
2. houses with the `2Story` style **or** central air conditioning,
3. houses that do **not** use `GasA` heating.

In [ ]:
# Filter 1: incorrect use of Python's "and"
# affordable_family_houses = df.loc[
#     (df["SalePrice"] < 180_000) and (df["BedroomAbvGr"] >= 2)
# ]

# Filter 2: missing parentheses
# two_storey_or_air = df.loc[
#     df["HouseStyle"] == "2Story" | df["CentralAir"] == "Y"
# ]

# Filter 3: wrong comparison operator for "not equal"
# houses_without_gasa = df.loc[df["Heating"] =! "GasA"]

### Verification challenge

After correcting the filters, verify each result using a different method. Examples:

- inspect the minimum or maximum of a relevant column,
- use `value_counts()`,
- use `query()` as an alternative,
- check whether an invalid value is present using `.any()`.

Simply displaying the first five rows is not sufficient verification.

## 8. Creating meaningful variables

A derived variable should represent a clearly defined concept and should use information from the correct point in time.

### Activity 5 — Find the conceptual bug

The following expression looks syntactically correct:

```python
df["HouseAge"] = 2026 - df["YearBuilt"]
```

Why is this a poor definition for analysing historical sales? Choose the more meaningful representation and create `HouseAgeAtSale`.

In [ ]:
df["HouseAgeAtSale"] = df["YrSold"] - df["YearBuilt"]

df.loc[:, ["YearBuilt", "YrSold", "HouseAgeAtSale"]].head()

### Verify the new variable

Check:

- its minimum and maximum,
- whether any values are negative,
- one row with the maximum age.

If you find an unexpected value, do not delete it automatically. First investigate it.

In [ ]:
df["HouseAgeAtSale"].describe()

### Activity 6 — From text to Boolean

Create a Boolean variable `HasCentralAir` from `CentralAir`.

Before writing code, answer:

- Which original value should map to `True`?
- How many distinct original values are present?
- How will you verify that the conversion is correct?

A concise starting expression is shown below. Complete and verify it.

In [ ]:
df["CentralAir"].value_counts(dropna=False)

# Modify the comparison:
df["HasCentralAir"] = df["CentralAir"].eq("...")

pd.crosstab(df["CentralAir"], df["HasCentralAir"], dropna=False)

## 9. Aggregation with `groupby`

Aggregation changes the unit of analysis. Instead of one row per house, the result may contain one row per building type, neighborhood, or heating category.

In [ ]:
df["Heating"].value_counts()

### Activity 7 — Choose a robust summary

Suppose we want to compare sale prices among building types.

Discuss:

1. Would the mean or median be more robust to very expensive houses?
2. Why is the group size necessary for interpretation?
3. Is comparing prices enough, or might living area also matter?

Complete the summary below by choosing appropriate aggregation functions.

In [ ]:
building_type_summary = (
    df.groupby("BldgType", observed=True)
      .agg(
          NumberOfHouses=("Id", "count"),
          TypicalPrice=("SalePrice", "..."),   # choose mean or median
          TypicalArea=("GrLivArea", "..."),    # choose and justify
      )
      .sort_values("TypicalPrice", ascending=False)
)

building_type_summary

### Interpretation checkpoint

Which building type has the highest typical price? Does this result prove that the building type itself causes the price difference? Name at least one possible confounding variable.

**A confounding variable is a third factor that is linked to both things you are comparing. It can make it look as if A causes B, when the real (or bigger) reason is that third factor.**

# 10. Visualisation

A graph is not the final result. Every graph should answer a question and should be followed by interpretation and verification.

We will compare:

1. **Matplotlib** — detailed control,
2. **Seaborn** — concise statistical graphics,
3. **Plotly Express** — interactive graphics.

## 10.1 Matplotlib — build and inspect a static plot

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["SalePrice"], bins=30, edgecolor="black")
plt.title("Distribution of house sale prices")
plt.xlabel("Sale price [USD]")
plt.ylabel("Number of houses")
plt.tight_layout()
plt.show()

### Activity 8 — Make two deliberate choices

Modify the histogram in two ways:

1. compare at least two different values of `bins`,
2. decide whether the x-axis would be easier to read in USD or thousands of USD.

Do not choose a setting only because it “looks nice”. Explain how the choice changes what the reader can see.

In [ ]:
# Modify the number of bins and, optionally, transform the displayed unit.
plot_values = df["SalePrice"]

plt.figure(figsize=(8, 5))
plt.hist(plot_values, bins=30, edgecolor="black")
plt.title("Distribution of house sale prices")
plt.xlabel("Sale price [USD]")
plt.ylabel("Number of houses")
plt.tight_layout()
plt.show()

## 10.2 Seaborn — choose a plot for a question

**Question:** How does sale price vary with living area, and does the pattern appear different for houses with and without central air?

Before running the code, answer what would you expect as a result.

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x="GrLivArea",
    y="SalePrice",
    hue="CentralAir",
    alpha=0.7,
)

plt.title("Sale price and living area by central air conditioning")
plt.xlabel("Above-ground living area [ft²]")
plt.ylabel("Sale price [USD]")
plt.tight_layout()
plt.show()

### Interpretation checkpoint
Write three sentences:

1. describe the main pattern,
2. identify unusual observations or region.


## 10.3 Plotly Express — investigate individual observations

Interactive plots are useful when hovering over individual observations changes what you can learn from the figure.

In [ ]:
fig = px.scatter(
    df,
    x="GrLivArea",
    y="SalePrice",
    color="CentralAir",
    hover_data=[
        "Id",
        "Neighborhood",
        "OverallQual",
        "YearBuilt",
        "BldgType",
    ],
    title="Interactive view of sale price and living area",
    labels={
        "GrLivArea": "Above-ground living area [ft²]",
        "SalePrice": "Sale price [USD]",
        "CentralAir": "Central air",
    },
    opacity=0.7,
)

fig.show()

### Activity 9 — Use interactivity for a purpose

Use zooming and hover information to select one unusual house.

Record:

- its `Id`,
- why it appears unusual,
- two other variables that may help explain its position.

Then modify `hover_data` by adding one variable that supports your investigation and removing one that does not.

# End-of-lab self-check
You do not submit this exercise for points. Before finishing, check that you can:

- [ ] explain the difference between `loc` and `iloc`,
- [ ] use logical operations in pandas,
- [ ] distinguish an identifier from a meaningful numerical feature,
- [ ] filter and select values,
- [ ] use min, max functions,
- [ ] create a plot in matplotlib, seaborn and plotly,
- [ ] approach missing values.

These skills will be used later in the semester project.